<a href="https://colab.research.google.com/github/kushal984762333-glitch/ibm-ds-capstone-spacex/blob/main/01-spacex-data-collection-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Data Collection with the SpaceX REST API
Requests launch data from `api.spacexdata.com/v4`, enriches it with rocket, launchpad, payload and core details, and saves a DataFrame.

In [1]:
import requests, pandas as pd, numpy as np
pd.set_option('display.max_columns',None)
spacex_url='https://api.spacexdata.com/v4/launches/past'
response=requests.get(spacex_url)
print(response.status_code)

522


In [9]:
import requests
import pandas as pd
import time

urls = [
    "https://api.spacexdata.com/v4/launches",
    "https://api.spacexdata.com/v4/launches/past"
]

data = None

for url in urls:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = pd.json_normalize(response.json())
        print("Data loaded successfully from:", url)
        break
    except (requests.RequestException, ValueError) as e:
        print(f"Failed to load from {url}: {e}")

if data is null or data.empty:
    print("Live API is unavailable. Loading static fallback dataset...")
    try:
        data = pd.read_json('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json')
        print("Static fallback dataset loaded successfully.")
    except Exception as e:
        print(f"Failed to load fallback dataset: {e}")

if data is not None:
    required_columns = [
        'rocket', 'payloads', 'launchpad',
        'cores', 'flight_number', 'date_utc'
    ]

    data = data[required_columns]

    data = data[data['cores'].map(len) == 1]
    data = data[data['payloads'].map(len) == 1]

    data['cores'] = data['cores'].map(lambda x: x[0])
    data['payloads'] = data['payloads'].map(lambda x: x[0])

    data['date'] = pd.to_datetime(data['date_utc']).dt.date

    data = data[
        data['date'] <= pd.Timestamp(2020, 11, 13).date()
    ]

    display(data.head())
else:
    print("Both API and fallback dataset are unavailable.")

Failed to load from https://api.spacexdata.com/v4/launches: HTTPSConnectionPool(host='api.spacexdata.com', port=443): Read timed out. (read timeout=10)
Failed to load from https://api.spacexdata.com/v4/launches/past: HTTPSConnectionPool(host='api.spacexdata.com', port=443): Read timed out. (read timeout=10)


NameError: name 'null' is not defined

In [7]:
BoosterVersion=[];PayloadMass=[];Orbit=[];LaunchSite=[];Longitude=[];Latitude=[]
Outcome=[];Flights=[];GridFins=[];Reused=[];Legs=[];LandingPad=[];Block=[];ReusedCount=[];Serial=[]

def getBoosterVersion(d):
    for x in d['rocket']:
        if x: BoosterVersion.append(requests.get('https://api.spacexdata.com/v4/rockets/'+str(x)).json()['name'])
def getLaunchSite(d):
    for x in d['launchpad']:
        if x:
            r=requests.get('https://api.spacexdata.com/v4/launchpads/'+str(x)).json()
            Longitude.append(r['longitude']);Latitude.append(r['latitude']);LaunchSite.append(r['name'])
def getPayloadData(d):
    for load in d['payloads']:
        if load:
            r=requests.get('https://api.spacexdata.com/v4/payloads/'+load).json()
            PayloadMass.append(r['mass_kg']);Orbit.append(r['orbit'])
def getCoreData(d):
    for core in d['cores']:
        if core['core'] is not None:
            r=requests.get('https://api.spacexdata.com/v4/cores/'+core['core']).json()
            Block.append(r['block']);ReusedCount.append(r['reuse_count']);Serial.append(r['serial'])
        else:
            Block.append(None);ReusedCount.append(None);Serial.append(None)
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight']);GridFins.append(core['gridfins']);Reused.append(core['reused']);Legs.append(core['legs']);LandingPad.append(core['landpad'])

getBoosterVersion(data);getLaunchSite(data);getPayloadData(data);getCoreData(data)

TypeError: 'NoneType' object is not subscriptable

In [8]:
launch_dict={'FlightNumber':list(data['flight_number']),'Date':list(data['date']),'BoosterVersion':BoosterVersion,'PayloadMass':PayloadMass,'Orbit':Orbit,'LaunchSite':LaunchSite,'Outcome':Outcome,'Flights':Flights,'GridFins':GridFins,'Reused':Reused,'Legs':Legs,'LandingPad':LandingPad,'Block':Block,'ReusedCount':ReusedCount,'Serial':Serial,'Longitude':Longitude,'Latitude':Latitude}
df=pd.DataFrame(launch_dict)
data_falcon9=df[df['BoosterVersion']!='Falcon 1'].reset_index(drop=True)
data_falcon9['FlightNumber']=list(range(1,data_falcon9.shape[0]+1))
data_falcon9['PayloadMass']=data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
print(data_falcon9.isnull().sum())
data_falcon9.to_csv('dataset_part_1.csv',index=False)
data_falcon9.head()

TypeError: 'NoneType' object is not subscriptable

**Fallback:** if the live API is unavailable, load the IBM static copy:
`pd.read_json('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json')`